# BRFSS 2015 - exploratory data analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/01_brfss_eda.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Dataset audit (shape, types, missing values, 24,206 repeated rows), class distribution (13.9% positive), feature distributions, Pearson/Spearman correlations, IQR outlier summary, class-wise statistics.

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


In [ ]:
import os

folders = [
    "Results",
    "Results/Figures",
    "Results/Tables",
    "Results/EDA"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders Created Successfully.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

plt.style.use('ggplot')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset Loaded Successfully")

In [ ]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

In [ ]:
shape_df = pd.DataFrame({
    "Rows":[df.shape[0]],
    "Columns":[df.shape[1]]
})

shape_df.to_csv("Results/Tables/dataset_shape.csv",index=False)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
info = pd.DataFrame({
    "Column":df.columns,
    "Data Type":df.dtypes.astype(str)
})

info.to_csv("Results/Tables/dataset_information.csv",index=False)

In [ ]:
missing = df.isnull().sum()

missing
missing.to_csv("Results/Tables/missing_values.csv")

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows :",duplicates)

In [ ]:
duplicate_df = pd.DataFrame({
    "Duplicate Rows":[duplicates]
})

duplicate_df.to_csv("Results/Tables/duplicate_records.csv",index=False)

In [ ]:
description = df.describe().T

description

In [ ]:
description.to_csv("Results/Tables/descriptive_statistics.csv")

In [ ]:
target = "Diabetes_binary"

In [ ]:
class_counts = df[target].value_counts().sort_index()

class_counts

In [ ]:
class_counts.to_csv("Results/Tables/class_distribution.csv")

In [ ]:
plt.figure(figsize=(6,5))

ax = sns.countplot(
    data=df,
    x=target,
    palette="Set2"
)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x()+0.25,p.get_height()+1000)
    )

plt.tight_layout()

plt.savefig("Results/Figures/class_distribution_bar.png",dpi=300)

plt.show()

In [ ]:
plt.figure(figsize=(7,7))

counts = df[target].value_counts()

labels = ["Non-Diabetic","Diabetic"]

plt.pie(
    counts,
    labels=labels,
    autopct="%1.2f%%",
    startangle=90,
    explode=(0.02,0.08),
    shadow=np.False_
)

plt.title("Distribution of Diabetes Classes")

plt.savefig(
    "Results/Figures/class_distribution_pie.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
percentage = round(
    df[target].value_counts(normalize=True)*100,
    2
)

distribution = pd.DataFrame({
    "Count":counts.values,
    "Percentage":percentage.values
},
index=["Non-Diabetic","Diabetic"])

distribution

In [ ]:
distribution.to_csv("Results/Tables/class_percentage.csv")

In [ ]:
# ==========================================
# Publication Quality Plot Settings
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

sns.set_context(
    "notebook",
    font_scale=1.2
)

soft_palette = [
    "#A8DADC",
    "#F4A261",
    "#E9C46A",
    "#B8C0FF",
    "#CDB4DB",
    "#95D5B2",
    "#F8C8DC",
    "#BDE0FE"
]

In [ ]:
# Numerical Features Distribution

features = [col for col in df.columns if col != target]

for feature in features:

    plt.figure(figsize=(7,5))

    sns.histplot(
        data=df,
        x=feature,
        bins=30,
        kde=True,
        color=soft_palette[0]
    )

    plt.title(f"Distribution of {feature}",
              fontsize=15,
              weight="bold")

    plt.xlabel(feature)

    plt.ylabel("Frequency")

    plt.tight_layout()

    plt.savefig(
        f"Results/Figures/{feature}_distribution.png",
        dpi=300
    )

    plt.show()

In [ ]:
for feature in features:

    plt.figure(figsize=(7,2.8))

    sns.boxplot(
        x=df[feature],
        color=soft_palette[4]
    )

    plt.title(f"Boxplot of {feature}",
              fontsize=14,
              weight='bold')

    plt.tight_layout()

    plt.savefig(
        f"Results/Figures/{feature}_boxplot.png",
        dpi=300
    )

    plt.show()

In [ ]:
summary = pd.DataFrame(index=features)

summary["Mean"] = df[features].mean()

summary["Median"] = df[features].median()

summary["Std"] = df[features].std()

summary["Minimum"] = df[features].min()

summary["Maximum"] = df[features].max()

summary["Skewness"] = df[features].skew()

summary["Kurtosis"] = df[features].kurt()

summary.round(3)

In [ ]:
summary.to_csv(
    "Results/Tables/feature_summary.csv"
)

In [ ]:
# ==========================================
# Pearson Correlation Heatmap
# ==========================================

corr_pearson = df.corr(method='pearson')

plt.figure(figsize=(18,14))

sns.heatmap(
    corr_pearson,
    cmap="YlGnBu",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar=True
)

plt.title(
    "Pearson Correlation Heatmap",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "Results/Figures/pearson_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
corr_pearson.to_csv(
    "Results/Tables/pearson_correlation_matrix.csv"
)


In [ ]:
# ==========================================
# Spearman Correlation Heatmap
# ==========================================

corr_spearman = df.corr(method='spearman')

plt.figure(figsize=(18,14))

sns.heatmap(
    corr_spearman,
    cmap="PuBuGn",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar=True
)

plt.title(
    "Spearman Correlation Heatmap",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "Results/Figures/spearman_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
corr_spearman.to_csv(
    "Results/Tables/spearman_correlation_matrix.csv"
)

In [ ]:
target_corr = (
    corr_pearson[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)

target_corr_df = pd.DataFrame({
    "Feature": target_corr.index,
    "Correlation": target_corr.values
})

display(target_corr_df)

target_corr_df.to_csv(
    "Results/Tables/feature_target_correlation.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(
    data=target_corr_df,
    y="Feature",
    x="Correlation",
    color="#A8DADC"
)

plt.title(
    "Correlation of Features with Diabetes",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Pearson Correlation Coefficient")

plt.ylabel("Features")

plt.tight_layout()

plt.savefig(
    "Results/Figures/feature_target_correlation.png",
    dpi=300
)

plt.show()

In [ ]:
# ==========================================
# Class-wise Feature Distribution
# ==========================================

features = [col for col in df.columns if col != target]

for feature in features:

    plt.figure(figsize=(7,5))

    sns.histplot(
        data=df,
        x=feature,
        hue=target,
        kde=True,
        palette=["#A8DADC", "#F4A261"],
        stat="density",
        common_norm=False,
        alpha=0.6
    )

    plt.title(
        f"{feature} Distribution by Diabetes Class",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(feature)
    plt.ylabel("Density")

    plt.tight_layout()

    plt.savefig(
        f"Results/Figures/{feature}_class_distribution.png",
        dpi=300
    )

    plt.show()

In [ ]:
# ==========================================
# Class-wise Feature Distribution
# Save all figures, display only first 5
# ==========================================

features = [col for col in df.columns if col != target]

display_limit = 5

for i, feature in enumerate(features):

    plt.figure(figsize=(7,5))

    sns.histplot(
        data=df,
        x=feature,
        hue=target,
        kde=True,
        palette=["#A8DADC", "#F4A261"],
        stat="density",
        common_norm=False,
        alpha=0.65
    )

    plt.title(
        f"{feature} Distribution by Diabetes Class",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(feature)
    plt.ylabel("Density")

    plt.tight_layout()

    # Save every figure
    plt.savefig(
        f"Results/Figures/{feature}_class_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )

    # Show only first five
    if i < display_limit:
        plt.show()
    else:
        plt.close()

print(f"\n✓ Saved {len(features)} class-wise distribution plots.")
print(f"✓ Displayed first {display_limit} plots only.")

In [ ]:
# ==========================================
# Outlier Summary using IQR
# ==========================================

outlier_summary = []

for feature in features:

    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[feature] < lower) | (df[feature] > upper)).sum()

    percentage = round((outliers / len(df)) * 100, 2)

    outlier_summary.append([
        feature,
        Q1,
        Q3,
        IQR,
        outliers,
        percentage
    ])

outlier_df = pd.DataFrame(
    outlier_summary,
    columns=[
        "Feature",
        "Q1",
        "Q3",
        "IQR",
        "Outliers",
        "Outlier (%)"
    ]
)

display(outlier_df)

In [ ]:
outlier_df.to_csv(
    "Results/Tables/outlier_summary.csv",
    index=False
)

print("✓ Outlier summary saved.")

In [ ]:
feature_type = []

for col in df.columns:

    feature_type.append([
        col,
        str(df[col].dtype),
        df[col].nunique(),
        df[col].min(),
        df[col].max()
    ])

feature_info = pd.DataFrame(
    feature_type,
    columns=[
        "Feature",
        "Data Type",
        "Unique Values",
        "Minimum",
        "Maximum"
    ]
)

display(feature_info)

In [ ]:
feature_info.to_csv(
    "Results/Tables/feature_information.csv",
    index=False
)

print("✓ Feature information saved.")

In [ ]:
# ==========================================
# Class-wise Summary Statistics
# ==========================================

class_summary = (
    df.groupby(target)
      .agg(['mean', 'median', 'std', 'min', 'max'])
      .round(2)
)

display(class_summary)

class_summary.to_csv(
    "Results/Tables/classwise_summary_statistics.csv"
)

print("✓ Class-wise summary statistics saved.")

In [ ]:
# ==========================================
# Ranked Feature Correlation
# ==========================================

rank_corr = (
    target_corr_df
    .reindex(target_corr_df["Correlation"].abs().sort_values(ascending=False).index)
    .reset_index(drop=True)
)

rank_corr.insert(0, "Rank", range(1, len(rank_corr)+1))

display(rank_corr)

rank_corr.to_csv(
    "Results/Tables/ranked_feature_correlation.csv",
    index=False
)

print("✓ Ranked correlation table saved.")

In [ ]:
# ==========================================
# Dataset Quality Report
# ==========================================

quality = pd.DataFrame({

    "Metric":[
        "Total Samples",
        "Total Features",
        "Missing Values",
        "Duplicate Records",
        "Target Classes"
    ],

    "Value":[
        len(df),
        len(df.columns)-1,
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df[target].nunique()
    ]

})

display(quality)

quality.to_csv(
    "Results/Tables/dataset_quality_report.csv",
    index=False
)

print("✓ Dataset quality report saved.")

In [ ]:
# ==========================================
# Save Clean Raw Dataset
# ==========================================

df.to_csv(
    "Results/EDA/clean_raw_dataset.csv",
    index=False
)

print("✓ Clean raw dataset saved.")